In [ ]:
import ee
import os
import geemap

# 1. KHỞI TẠO VÀ KẾT NỐI
ee.Authenticate() # Bỏ comment nếu chạy lần đầu
ee.Initialize(project='geemap-mekong-483717') # Thay bằng Project ID của bạn

print("✅ Đã kết nối Google Earth Engine")

# 2. ĐỊNH NGHĨA ROI (ĐỒNG BẰNG SÔNG CỬU LONG)
mekong_provinces = [
    'An Giang', 'Bac Lieu', 'Ben Tre', 'Ca Mau', 'Can Tho city',
    'Dong Thap', 'Hau Giang', 'Kien Giang', 'Long An',
    'Soc Trang', 'Tien Giang', 'Tra Vinh', 'Vinh Long'
]
roi = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.inList('ADM1_NAME', mekong_provinces))
roi_geom = roi.geometry()

# 3. HÀM XỬ LÝ ẢNH SENTINEL-2
def mask_s2_clouds(image):
    """Hàm lọc mây cho Sentinel-2 sử dụng dải QA60"""
    qa = image.select('QA60')
    # Bit 10 và 11 là mây dày và mây ti (cirrus)
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    # Cả hai cờ này phải bằng 0 thì mới là trời trong
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
        .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

def add_indices(image):
    """Tính toán NDVI và NDWI"""
    # NDVI = (NIR - Red) / (NIR + Red) = (B8 - B4) / (B8 + B4)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    # NDWI = (Green - NIR) / (Green + NIR) = (B3 - B8) / (B3 + B8)
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    return image.addBands([ndvi, ndwi])

# 4. TRÍCH XUẤT DỮ LIỆU ĐỊNH KỲ (Ví dụ: 3 tháng đầu năm 2024)
YEAR = 2023
months = [7] # Lấy 3 tháng làm ví dụ PoC

out_folder = 'Sentinel2_Periodic_DBSCL'

print("⏳ Bắt đầu tạo tác vụ tải dữ liệu Sentinel-2...")

for month in months:
    # Lấy ngày đầu tháng và ngày cuối tháng
    start_date = ee.Date.fromYMD(YEAR, month, 1)
    end_date = start_date.advance(1, 'month')

    # Ngày đại diện cho file (dùng ngày 15 giữa tháng làm mốc cho ảnh composite)
    date_str = f"{YEAR}-{month:02d}-15"

    # Truy vấn collection Sentinel-2 Surface Reflectance
    s2_collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(roi_geom) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \
        .map(mask_s2_clouds) \
        .map(add_indices)

    # Lấy giá trị trung vị (median) của tháng để có ảnh sạch nhất, sau đó cắt theo ROI
    s2_composite = s2_collection.median().clip(roi_geom)

    # Lấy riêng 2 dải màu cần thiết để xuất
    ndvi_img = s2_composite.select('NDVI')
    ndwi_img = s2_composite.select('NDWI')

    # --- XUẤT TASK LÊN GOOGLE DRIVE ---
    # Xuất NDVI
    task_ndvi = ee.batch.Export.image.toDrive(
        image=ndvi_img,
        description=f'Task_NDVI_{date_str}',
        folder=out_folder,
        fileNamePrefix=f'NDVI_{date_str}', # Tên file cực kỳ quan trọng cho Regex
        scale=100, # Đặt 100m để test PoC cho nhanh, chạy thật có thể set 10m hoặc 30m
        region=roi_geom,
        maxPixels=1e13
    )
    task_ndvi.start()

    # Xuất NDWI
    task_ndwi = ee.batch.Export.image.toDrive(
        image=ndwi_img,
        description=f'Task_NDWI_{date_str}',
        folder=out_folder,
        fileNamePrefix=f'NDWI_{date_str}',
        scale=100,
        region=roi_geom,
        maxPixels=1e13
    )
    task_ndwi.start()

    print(f"✅ Đã đẩy tác vụ tháng {month:02d}/{YEAR} (Date: {date_str}) lên GEE Tasks.")

print("\n🎉 Xong! Vui lòng vào tab Tasks trên Google Earth Engine Code Editor để theo dõi tiến độ.")